# Session 11: The Mood Machine Challenge

Click a cell and press **Shift + Enter** to run it. Work from top to bottom.
Cells marked **YOUR TURN** are for you to finish. Run the first cell straight away: it draws 15,000 faces.

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import matplotlib.pyplot as plt
from emoji_faces import make_face, CLASSES

if torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator()
else:
    device = torch.device("cpu")
print("Using", device)

class EmojiFaces(Dataset):
    def __init__(self, how_many, first_seed, transform=None):
        self.faces = []
        for seed in range(first_seed, first_seed + how_many):
            picture, mood = make_face(seed)
            if transform:
                picture = transform(picture)
            self.faces.append((picture, mood))

    def __len__(self):
        return len(self.faces)

    def __getitem__(self, idx):
        return self.faces[idx]

get_ready = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

print("Drawing faces... (about a minute)")
train_faces = EmojiFaces(12000, first_seed=0, transform=get_ready)
test_faces = EmojiFaces(3000, first_seed=1_000_000, transform=get_ready)
train_loader = DataLoader(train_faces, batch_size=64, shuffle=True)
test_loader = DataLoader(test_faces, batch_size=64)
print(len(train_faces), "training faces and", len(test_faces), "test faces")

class MoodNet(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.flatten = nn.Flatten()
        self.hidden1 = nn.Linear(3 * 32 * 32, hidden)
        self.hidden2 = nn.Linear(hidden, 128)
        self.output = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        return self.output(x)

def train_one_epoch(model, loader, loss_fn, optimiser):
    model.train()
    for faces, moods in loader:
        faces, moods = faces.to(device), moods.to(device)
        guesses = model(faces)            # 1. make guesses
        loss = loss_fn(guesses, moods)    # 2. how wrong were they?
        optimiser.zero_grad()             # 3. wipe old gradients
        loss.backward()                   # 4. find new gradients (back propagation)
        optimiser.step()                  # 5. nudge the weights
    return loss.item()

def check_accuracy(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for faces, moods in loader:
            faces, moods = faces.to(device), moods.to(device)
            guesses = model(faces).argmax(dim=1)
            correct += (guesses == moods).sum().item()
    return 100 * correct / len(loader.dataset)

## 1. Overfitting in Action
A Mood Machine trained on only 500 faces for 20 epochs. Watch the two scores.

In [ ]:
small_set = torch.utils.data.Subset(train_faces, range(500))
small_loader = DataLoader(small_set, batch_size=64, shuffle=True)
torch.manual_seed(1)
net = MoodNet().to(device)
opt = torch.optim.Adam(net.parameters(), lr=0.001)
for epoch in range(1, 21):
    train_one_epoch(net, small_loader, nn.CrossEntropyLoss(), opt)
    if epoch % 4 == 0:
        print("epoch", epoch, " training", round(check_accuracy(net, DataLoader(small_set, batch_size=256)), 1), "%",
              "  test", round(check_accuracy(net, test_loader), 1), "%")

## 2. The Challenge
Change one setting at a time. Record every run.

In [ ]:
def train_mood_machine(hidden=256, learning_rate=0.001, epochs=3, training_faces=12000, seed=1):
    torch.manual_seed(seed)
    subset = torch.utils.data.Subset(train_faces, range(training_faces))
    loader = DataLoader(subset, batch_size=64, shuffle=True)
    net = MoodNet(hidden=hidden).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=learning_rate)
    for _ in range(epochs):
        train_one_epoch(net, loader, nn.CrossEntropyLoss(), opt)
    train_acc = round(check_accuracy(net, DataLoader(subset, batch_size=256)), 1)
    test_acc = round(check_accuracy(net, test_loader), 1)
    print("training", train_acc, "%   test", test_acc, "%")
    return net

baseline = train_mood_machine()

In [ ]:
# YOUR TURN: one change at a time. Some ideas:
#   train_mood_machine(epochs=6)
#   train_mood_machine(hidden=512)
#   train_mood_machine(learning_rate=0.003)
#   train_mood_machine(training_faces=3000)
#   train_mood_machine(seed=2)     # how much do scores vary with nothing else changed?